# 01 · NEU Steel Surface Defects - EDA

**Arkon Manufacturing | Department: Steel Rolling**

Dataset: 6 classes of steel surface defects (classification + detection)
- Classes: `crazing, inclusion, patches, pitted_surface, rolled-in_scale, scratches`
- Structure: `train/images/[class]/` + `train/annotations/[class]/`
- Annotations: Pascal VOC XML (bounding boxes)

In [ ]:
import sys
from pathlib import Path

_nb_root = Path('../..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)
print('arkon_utils loaded ✓')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET
from collections import Counter
import pandas as pd

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
ASSETS = 'cv/neu'


In [ ]:
DATA_DIR   = Path('../../../data/04_neu/raw')
TRAIN_IMGS = DATA_DIR / 'train' / 'images'
TRAIN_ANN  = DATA_DIR / 'train' / 'annotations'
VAL_IMGS   = DATA_DIR / 'validation' / 'images'
VAL_ANN    = DATA_DIR / 'validation' / 'annotations'

assert DATA_DIR.exists(), f'Not found: {DATA_DIR}'
CLASSES = sorted([d.name for d in TRAIN_IMGS.iterdir() if d.is_dir()])
print(f'Classes ({len(CLASSES)}): {CLASSES}')

## 1. Image Count per Class

In [ ]:
rows = []
for split_name, img_root in [('train', TRAIN_IMGS), ('validation', VAL_IMGS)]:
    for cls in CLASSES:
        imgs = list((img_root / cls).glob('*.jpg')) + list((img_root / cls).glob('*.bmp'))
        rows.append({'split': split_name, 'class': cls, 'count': len(imgs)})

df = pd.DataFrame(rows)
print(df.pivot(index='class', columns='split', values='count'))
print(f"\nTotal: {df['count'].sum()} images")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df_train = df[df['split']=='train'].sort_values('class')
df_val   = df[df['split']=='validation'].sort_values('class')
x = range(len(CLASSES))
width = 0.35
ax.bar([i - width/2 for i in x], df_train['count'], width, label='Train', color='steelblue')
ax.bar([i + width/2 for i in x], df_val['count'],   width, label='Val',   color='coral')
ax.set_xticks(list(x)); ax.set_xticklabels(CLASSES, rotation=20)
ax.set_ylabel('Images'); ax.legend()
plt.title('NEU - Class Distribution')
plt.tight_layout()
save_figure(fig, 'cv_neu_plot_1', subfolder='cv/neu')
plt.show()

## 2. Sample Images per Defect Class

In [ ]:
fig, axes = plt.subplots(2, len(CLASSES), figsize=(16, 6))
for col, cls in enumerate(CLASSES):
    for row, img_root in enumerate([TRAIN_IMGS, VAL_IMGS]):
        imgs = list((img_root / cls).glob('*.jpg')) + list((img_root / cls).glob('*.bmp'))
        if imgs:
            img = Image.open(imgs[0]).convert('RGB')
            axes[row][col].imshow(img, cmap='gray')
        axes[row][col].axis('off')
        if row == 0: axes[row][col].set_title(cls, fontsize=8)
axes[0][0].set_ylabel('Train', fontsize=10)
axes[1][0].set_ylabel('Val',   fontsize=10)
plt.suptitle('NEU - One Sample per Class')
plt.tight_layout()
save_figure(fig, 'cv_neu_plot_2', subfolder='cv/neu')
plt.show()

## 3. Annotation Analysis - bounding boxes (Pascal VOC XML)

In [ ]:
def parse_voc_xml(xml_path: Path) -> list:
    tree = ET.parse(xml_path)
    root = tree.getroot()
    boxes = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        boxes.append({'class': name, 'xmin': xmin, 'ymin': ymin,
                      'xmax': xmax, 'ymax': ymax,
                      'width': xmax-xmin, 'height': ymax-ymin})
    return boxes

# Sample: parse a few XMLs
all_boxes = []
for cls in CLASSES:
    ann_dir = TRAIN_ANN / cls
    if ann_dir.exists():
        for xml_f in list(ann_dir.glob('*.xml'))[:20]:
            all_boxes.extend(parse_voc_xml(xml_f))

df_boxes = pd.DataFrame(all_boxes)
if not df_boxes.empty:
    print(df_boxes.groupby('class')[['width','height']].describe().round(1))

## Summary

| Parameter | Value |
|---|---|
| Task | 6-class Classification (+ Detection) |
| Annotations | Pascal VOC XML (bounding boxes) |
| Note | Small images (200×200) |

➡️ **Next step:** `02_neu_preprocessing.ipynb`